In [1]:
import pandas as pd

import re

In [4]:
df = pd.read_parquet('../data/raw/neo4j-2024v1/train-00000-of-00001.parquet')
df

,question,schema,cypher,data_source,instance_id,database_reference_alias
0,Which 3 countries have the most entities linke...,Node properties:\n- **Country**\n - `location...,MATCH (f:Filing)-[:BENEFITS]->(e:Entity)-[:COU...,neo4jLabs_synthetic_gpt4o,instance_id_41185,neo4jlabs_demo_db_fincen
1,What are the names of the first 3 organization...,Node properties:\n- **Person**\n - `name`: ST...,MATCH (o:Organization)-[:HAS_CEO]->(ceo:Person...,neo4jLabs_synthetic_gpt4turbo,instance_id_26598,neo4jlabs_demo_db_companies
2,List the names of the games played by streams ...,Node properties:\n- **Stream**\n - `createdAt...,MATCH (s:Stream)-[:PLAYS]->(g:Game) WHERE s.to...,neo4jLabs_synthetic_gemini,instance_id_34035,neo4jlabs_demo_db_twitch
3,For each Article find its abstract and the cou...,Graph schema: Relevant node labels and their p...,MATCH (n:Article) -[:HAS_KEY]->(m:Keyword) WIT...,neo4jLabs_functional_cypher,instance_id_3914,None
4,Find the Author for which first_name starts wi...,Graph schema: Relevant node labels and their p...,MATCH (n:Author) WHERE n.first_name STARTS WIT...,neo4jLabs_functional_cypher,instance_id_14645,None
...,...,...,...,...,...,...
39549,Who are the top 5 users with x-coordinate valu...,Node properties:\n- **User**\n - `label`: STR...,MATCH (u:User) WHERE u.x < -5000 RETURN u.labe...,neo4jLabs_synthetic_gpt4o,instance_id_40775,neo4jlabs_demo_db_bluesky
39550,Find the shortest path between Journal where j...,Graph schema: Relevant node labels and their p...,MATCH p=shortestPath((a:Journal{journal_id:'f7...,neo4jLabs_functional_cypher,instance_id_6588,None
39551,Return the first_name for Author combined with...,Graph schema: Relevant node labels and their p...,MATCH (n:Author) RETURN n.first_name AS Record...,neo4jLabs_functional_cypher,instance_id_16140,None
39552,Which 3 users have the highest rate of answere...,Node properties:\n- **Question**\n - `favorit...,MATCH (u:User)-[:ASKED]->(q:Question) WHERE q....,neo4jLabs_synthetic_gpt4o,instance_id_40363,neo4jlabs_demo_db_buzzoverflow


In [5]:
# count and shows unique database_reference_alias column values
df['database_reference_alias'].value_counts(dropna=False)

database_reference_alias
None                                 17461
neo4jlabs_demo_db_eoflix              2512
neo4jlabs_demo_db_companies           2422
neo4jlabs_demo_db_recommendations     2130
neo4jlabs_demo_db_movies              1949
neo4jlabs_demo_db_northwind           1669
neo4jlabs_demo_db_twitch              1604
neo4jlabs_demo_db_grandstack          1492
neo4jlabs_demo_db_fincen              1347
neo4jlabs_demo_db_gameofthrones       1296
neo4jlabs_demo_db_twitter             1284
neo4jlabs_demo_db_buzzoverflow        1106
neo4jlabs_demo_db_network             1094
neo4jlabs_demo_db_offshoreleaks        921
neo4jlabs_demo_db_stackoverflow2       867
neo4jlabs_demo_db_bluesky              388
neo4jlabs_demo_db_openstreetmap         10
neo4jlabs_demo_db_stackoverflow          2
Name: count, dtype: int64

In [ ]:
# print how many rows have database_reference_alias
print(f"Rows with database_reference_alias: {df['database_reference_alias'].notnull().sum()}")

In [ ]:
# filter rows that starts with "Node properties:"
mask = df['schema'].str.startswith("Node properties:")
to_remove = df[~mask]
df = df[mask]
print(f"Removing {len(to_remove)} rows that do not start with 'Node properties:'")
print(f"Remaining rows: {len(df)}")
print("Removed rows:")
to_remove

In [ ]:
# replace each ` character with nothing
df['schema'] = df['schema'].str.replace('`', '')

In [ ]:
# schema_regex =r'Node properties:\n(?P<nodes>(?:(?:.*)\n)+)Relationship properties:\n(?P<rel_properties>(?:(?:.*)\n)+)The relationships:\n(?P<relationships>(?:(?:.*)\n?)+)'
# node_regex = r'- \*\*(?P<node>[a-zA-Z\_]+)\*\*\n(?P<properties>(?:  - (?:.+)\n)+)'
# properties_regex = r'  - `(?P<property>[a-zA-Z\_]+)`: (?P<property_type>[a-zA-Z\_]+) (?P<example>\s*Example: \".+\")?(?:Min: (?P<min>.+), Max: (?P<max>.+))?\n?'

In [ ]:
removed_groups_regex = re.compile(r'\?P<\w+>')

In [ ]:
# property_regex = r'  - `(?P<property_name>.+)`: (?P<property_type>\w+) (?P<example>\s*Example: \".*\")?(?P<extra>.+)?\n?'
property_regex = r'  - (?P<property_name>\w+): (?P<property_type>\w+) (?P<extra>.+)?\n?'
# replace all named groups in node_regex with ?: to make them non-capturing

# temp_single_prop_regex = removed_groups_regex.sub('?:', property_regex)
entity_regex = rf'- \*\*(?P<entity>\w+)\*\*\n+(?P<properties>(?:{property_regex})+)'
cleaned_entity_regex = removed_groups_regex.sub('?:', entity_regex)
print(cleaned_entity_regex, end="\n\n")

# temp_node_regex = removed_groups_regex.sub('?:', entity_regex)
# schema_regex =rf'Node properties:\n(?P<nodes>(?:{entity_regex}\n+)+)Relationship properties:\n(?P<rel_properties>(?:(?:.*)\n)+)The relationships:\n(?P<relationships>(?:(?:.*)\n?)+)'

schema_regex =rf'Node properties:\n+(?P<nodes>(?:{cleaned_entity_regex}\n+)+)Relationship properties:\n+(?P<rel_properties>(?:{cleaned_entity_regex}\n+)*)The relationships:\n+(?P<relationships>(?:(?:.*)\n?)+)'

# temp_schema_regex = removed_groups_regex.sub('?:', schema_regex)

print(schema_regex)

# Schema regex

754 da rimuovere

In [ ]:
# remove rows that do not match with the context regex
to_keep = df['schema'].apply(lambda x: re.match(schema_regex, x) is not None)
to_remove = df[~to_keep]
print(f"Removing {len(to_remove)} rows that do not match the schema regex")
to_remove

In [ ]:
entity_regex2 = r'\w+ (?P<properties_dict>\{(?:(?:\w+\:\s+\w+),?\s*)+\})'
cleaned_entity_regex2 = removed_groups_regex.sub('?:', entity_regex2)
print(cleaned_entity_regex2, end="\n\n")
schema2_regex =rf'Node properties:\n+(?P<nodes>(?:{cleaned_entity_regex2}\n+)+)Relationship properties:\n+(?P<rel_properties>(?:{cleaned_entity_regex2}\n+)*)The relationships:\n+(?P<relationships>(?:(?:.*)\n?)+)'
print(schema2_regex)

In [ ]:
to_keep2 = to_remove['schema'].apply(lambda x: re.match(schema2_regex, x) is not None)
to_remove2 = to_remove[~to_keep2]
print(f"Removing {len(to_remove2)} rows that do not match the schema2 regex")
to_remove2

In [ ]:
df = df[to_keep]
df